Poisson Equation
===

Import Netgen/NGSolve Python modules:

In [1]:
from ngsolve import *
from ngsolve.webgui import Draw as _Draw

def Draw(*args, height="350px", **kwargs):
    return _Draw(*args, height=height, **kwargs)

import ipywidgets
# Force widget comm initialization
from IPython.display import display
ipywidgets.Widget.close_all()



The unit_square is a predefined domain, use Netgen to generate a mesh:

In [2]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
Draw (mesh);

WebGuiWidget(layout=Layout(height='350px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Define a finite element space on that mesh. 

In [3]:
fes = H1(mesh, order=3, dirichlet="left|right|bottom|top")
print ("ndof =", fes.ndof)

ndof = 274


In [4]:
print (mesh.GetBoundaries())

('bottom', 'right', 'top', 'left')


Define linear and bilinear-forms. 

$$
a(u,v) = \int \nabla u \nabla v
\qquad \text{and} \qquad
f(v) = \int f v
$$

Forms are expressed in terms of trial and test-functions:

In [5]:
u = fes.TrialFunction()
v = fes.TestFunction()

f = LinearForm(fes)
f += 32*(y*(1-y)+x*(1-x))*v*dx

a = BilinearForm(fes)
a += grad(u)*grad(v)*dx

a.Assemble()
f.Assemble();

In [6]:
print(f.vec)
print(a.mat)

 0.0196267
 0.0196267
 0.0196267
 0.0309262
 0.159433
 0.219303
 0.181239
 0.137783
 0.215831
 0.164775
 0.143975
 0.119435
 0.172482
 0.170324
 0.181616
 0.0965241
 0.0926509
 0.163062
 0.214707
 0.15821
 0.640245
 0.537739
 0.45428
 0.423631
 0.369049
 0.345279
 0.299249
 0.570043
 0.365346
 0.158316
 0.345473
 0.513418
 0.508978
 0.611025
 0.42921
 0.53896
 0.339249
 0.53602
 -0.00291556
 -0.000294603
 -0.00291556
 -0.000294603
 -0.00291556
 -0.000294603
 -0.00291556
 -0.000294603
 -0.00291556
 -0.000294603
 -0.00291556
 -0.000294603
 -0.00218081
 -0.00019139
 -0.00214635
 -0.00018831
 -0.00485111
 -0.000555929
 -0.00797493
 -0.000179369
 -0.0119854
 6.01459e-07
 -0.0184261
 -0.000948946
 -0.00730748
 -1.50672e-06
 -0.0211592
 -0.00056396
 -0.0201285
 -0.000658461
 -0.0057403
 0.000127886
 -0.0177395
 -0.000594187
 -0.0159163
 -0.000386791
 -0.0115799
 -1.49723e-05
 -0.0152841
 -0.000762461
 -0.00643943
 -0.000132993
 -0.018645
 -0.00084282
 -0.0169044
 -0.000848585
 -0.00596394
 3.

Solve the problem:

In [7]:
gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs()) * f.vec

Plot the solution:

In [8]:
Draw (gfu, mesh, height="300px");

WebGuiWidget(layout=Layout(height='300px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [9]:
P0 = VectorL2(mesh, order=0)
gf_grad = GridFunction(P0)
gf_grad.Set(-grad(gfu))
Draw(gf_grad, mesh, "Flux", vectors= { "grid_size" : 40});

WebGuiWidget(layout=Layout(height='350px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Calculate error:

In [10]:
exact = 16*x*(1-x)*y*(1-y)
print ("L2-error:", sqrt(Integrate((gfu-exact)**2, mesh)))

L2-error: 5.810212954077356e-05


In [11]:
vtk = VTKOutput(mesh, coefs=[gfu, -grad(gfu)[0]], names=["sol", "grad_sol"], filename="output")
vtk.Do()

'output'

In [12]:
print(-grad(gfu))
mesh.ne

-1*(coef N6ngcomp31GridFunctionCoefficientFunctionE, real, dim=2
)


54

In [13]:
help(VTKOutput)


Help on class VTKOutput in module ngsolve.comp:

class VTKOutput(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      VTKOutput
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  Do(...)
 |      Do(*args, **kwargs)
 |      Overloaded function.
 |
 |      1. Do(self: ngsolve.comp.VTKOutput, time: typing.SupportsFloat = -1, vb: ngsolve.comp.VorB = <VorB.VOL: 0>) -> str
 |
 |
 |      Write mesh and fields to file. When called several times on the same object
 |      an index is added to the output file name. A meta file (.pvd) is written
 |      (unless in legacy mode).
 |
 |      Returns string of the output filename.
 |
 |      Parameters:
 |
 |      time :
 |        associate a time to the current output
 |
 |      vb: VOL_or_BND (default VOL)
 |        defines if output is done on the volume (VOL) or surface mesh (BND).
 |                  .
 |
 |      2. Do(self: ngsolve.comp.VTKOutput, time: typing.SupportsFloat =